# CV05 — Hiệp: Tinh chỉnh TimeSeriesSplit & Benchmark Random Forest

**Thành viên phụ trách:** Hiệp  
**Đầu vào:** `data/processed/train.csv`, `test.csv` (CV01)  
**Đầu ra:**  
- `outputs/tables/cv05_best_hyperparameters.csv`  
- `outputs/tables/cv05_benchmark_summary.csv`  
- `outputs/figures/cv05_benchmark_comparison.png`  
- `outputs/models/rf_benchmark.joblib`

---

## 1. Cấu hình

In [ ]:
import sys, time, joblib
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from xgboost import XGBRegressor

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
TRAIN_DATA   = PROJECT_ROOT / 'data' / 'processed' / 'train.csv'
TEST_DATA    = PROJECT_ROOT / 'data' / 'processed' / 'test.csv'
MODELS_DIR   = PROJECT_ROOT / 'outputs' / 'models'
FIGURES_DIR  = PROJECT_ROOT / 'outputs' / 'figures'
TABLES_DIR   = PROJECT_ROOT / 'outputs' / 'tables'

TARGET_COL = 'AEP_MW'
CONFIG = {
    'cv_splits':    5,
    'random_state': 42,
    'param_grid': {
        'max_depth':     [3, 5, 7],
        'learning_rate': [0.01, 0.05, 0.1],
        'n_estimators':  [300, 500],
        'subsample':     [0.8, 1.0],
    },
    'rf_params': {
        'n_estimators': 300,
        'max_depth':    None,
        'n_jobs':       -1,
        'random_state': 42,
    },
}
print('✅ Cấu hình hoàn tất')

## 2. Nạp dữ liệu từ CV01

In [ ]:
def load_df(path):
    df = pd.read_csv(path)
    df['Datetime'] = pd.to_datetime(df['Datetime'])
    df.set_index('Datetime', inplace=True)
    return df

train_df = load_df(TRAIN_DATA)
test_df  = load_df(TEST_DATA)

features = [c for c in train_df.columns if c != TARGET_COL]
X_train, y_train = train_df[features], train_df[TARGET_COL].astype(float)
X_test,  y_test  = test_df[features],  test_df[TARGET_COL].astype(float)

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')

## 3. Tinh chỉnh siêu tham số XGBoost bằng TimeSeriesSplit (Bài tập nâng cao 3)

In [ ]:
print(f'GridSearchCV với TimeSeriesSplit(n_splits={CONFIG["cv_splits"]})')
print('⚠ Lưu ý: KHÔNG dùng KFold(shuffle=True) — bắt buộc giữ thứ tự thời gian!')

tscv = TimeSeriesSplit(n_splits=CONFIG['cv_splits'])
base_model = XGBRegressor(
    objective='reg:squarederror',
    random_state=CONFIG['random_state'],
    n_jobs=-1,
    tree_method='hist',
)

n_comb = int(np.prod([len(v) for v in CONFIG['param_grid'].values()]))
print(f'Số tổ hợp: {n_comb} x {CONFIG["cv_splits"]} folds = {n_comb * CONFIG["cv_splits"]} lần fit')

t0 = time.perf_counter()
search = GridSearchCV(
    estimator=base_model,
    param_grid=CONFIG['param_grid'],
    cv=tscv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1, refit=False, verbose=1
)
search.fit(X_train, y_train)
grid_time = time.perf_counter() - t0

best_params  = dict(search.best_params_)
best_cv_mae  = float(-search.best_score_)
print(f'\n⏱  Thời gian tìm kiếm: {grid_time:.1f}s')
print(f'🏆  Bộ tham số tối ưu: {best_params}')
print(f'📉  CV MAE tốt nhất  : {best_cv_mae:.3f} MW')

In [ ]:
# Lưu bảng siêu tham số
hyper_table = pd.DataFrame({'hyperparameter': list(best_params.keys()),
                             'best_value':    list(best_params.values())})
hyper_table.to_csv(TABLES_DIR / 'cv05_best_hyperparameters.csv', index=False)
print('✅ Đã lưu cv05_best_hyperparameters.csv')
hyper_table

## 4. Huấn luyện lại XGBoost (tuned) & Benchmark Random Forest (Bài tập nâng cao 4)

In [ ]:
# --- XGBoost Tuned ---
t0 = time.perf_counter()
tuned_xgb = XGBRegressor(objective='reg:squarederror', random_state=42,
                          n_jobs=-1, tree_method='hist', **best_params)
tuned_xgb.fit(X_train, y_train)
xgb_train_time = time.perf_counter() - t0

# --- Random Forest ---
t0 = time.perf_counter()
rf_model = RandomForestRegressor(**CONFIG['rf_params'])
rf_model.fit(X_train, y_train)
rf_train_time = time.perf_counter() - t0

print(f'XGBoost (tuned) train time : {xgb_train_time:.1f}s')
print(f'Random Forest   train time : {rf_train_time:.1f}s')

In [ ]:
def reg_metrics(y_true, y_pred):
    mae  = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mape = float(np.mean(np.abs((y_true - y_pred) / y_true)) * 100)
    r2   = float(r2_score(y_true, y_pred))
    return mae, rmse, mape, r2

y_arr = y_test.to_numpy()

xgb_pred = tuned_xgb.predict(X_test).astype(float)
rf_pred  = rf_model.predict(X_test).astype(float)

xgb_m = reg_metrics(y_arr, xgb_pred)
rf_m  = reg_metrics(y_arr, rf_pred)

# Inference speed
t0 = time.perf_counter(); tuned_xgb.predict(X_test); xgb_ms = (time.perf_counter()-t0)*1000
t0 = time.perf_counter(); rf_model.predict(X_test);  rf_ms  = (time.perf_counter()-t0)*1000

# Overfit gap
xgb_train_mae = float(mean_absolute_error(y_train, tuned_xgb.predict(X_train)))
rf_train_mae  = float(mean_absolute_error(y_train, rf_model.predict(X_train)))

comparison = pd.DataFrame({
    'Model':              ['XGBoost (tuned)', 'Random Forest'],
    'MAE (MW)':          [xgb_m[0], rf_m[0]],
    'RMSE (MW)':         [xgb_m[1], rf_m[1]],
    'MAPE (%)':          [xgb_m[2], rf_m[2]],
    'R2 Score':          [xgb_m[3], rf_m[3]],
    'Train Time (s)':    [xgb_train_time, rf_train_time],
    'Inference (ms)':    [xgb_ms, rf_ms],
    'Train MAE (MW)':    [xgb_train_mae, rf_train_mae],
    'Overfit Gap (MW)':  [xgb_m[0]-xgb_train_mae, rf_m[0]-rf_train_mae],
})
comparison.to_csv(TABLES_DIR / 'cv05_benchmark_summary.csv', index=False)
print('✅ Đã lưu cv05_benchmark_summary.csv')
comparison.round(4)

## 5. Biểu đồ so sánh Benchmark

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics_cols = [('MAE (MW)', 'So sánh MAE (MW)'), ('RMSE (MW)', 'So sánh RMSE (MW)'), ('Train Time (s)', 'Thời gian huấn luyện (s)')]
colors = ['#4C78A8', '#F58518']

for ax, (col, title), color in zip(axes, metrics_cols, colors):
    bars = ax.bar(comparison['Model'], comparison[col], color=color)
    ax.set_title(title, fontsize=12)
    ax.grid(axis='y', alpha=0.3)
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:,.2f}', xy=(bar.get_x() + bar.get_width()/2, h),
                    xytext=(0, 3), textcoords='offset points', ha='center', va='bottom')

axes[0].set_ylabel('MW'); axes[1].set_ylabel('MW'); axes[2].set_ylabel('giây')
fig.suptitle('CV05: XGBoost (tuned) vs Random Forest — Benchmark so sánh', fontsize=13, fontweight='bold')
fig.tight_layout()

fig_path = FIGURES_DIR / 'cv05_benchmark_comparison.png'
fig.savefig(fig_path, dpi=150)
plt.show()
print(f'✅ Đã lưu: {fig_path}')

## 6. Lưu model Random Forest

In [ ]:
rf_path = MODELS_DIR / 'rf_benchmark.joblib'
joblib.dump(rf_model, rf_path)
print(f'✅ Checkpoint RF đã lưu: {rf_path}  ({rf_path.stat().st_size/1024:.0f} KB)')

## 7. Nhận xét kết quả

### 7.1 TimeSeriesSplit vs KFold
- `TimeSeriesSplit(n_splits=5)` chia theo thứ tự thời gian tuần tự (expanding window), đảm bảo fold sau chứa dữ liệu muộn hơn fold trước.
- `KFold(shuffle=True)` vi phạm nguyên tắc chuỗi thời gian vì có thể đưa dữ liệu tương lai vào tập train → **Data Leakage**.

### 7.2 XGBoost vs Random Forest
| Tiêu chí | XGBoost (tuned) | Random Forest |
|:---|:---|:---|
| Độ chính xác | ✅ Thường tốt hơn | Tốt nhưng thua nhẹ |
| Tốc độ train | ✅ Nhanh hơn với `hist` | Chậm hơn (song song) |
| Inference | ✅ Nhanh hơn | Chậm hơn |
| Overfitting | Tốt (early stopping) | RF ít overfit hơn nếu RF train MAE thấp |

## ✅ CV05 Hoàn tất

| Sản phẩm | Đường dẫn |
|:---|:---|
| Siêu tham số tối ưu | `outputs/tables/cv05_best_hyperparameters.csv` |
| Bảng benchmark | `outputs/tables/cv05_benchmark_summary.csv` |
| Biểu đồ so sánh | `outputs/figures/cv05_benchmark_comparison.png` |
| RF checkpoint | `outputs/models/rf_benchmark.joblib` |